# Rangkuman Kinerja Proyek NIDS Lintas-Jaringan (SFM + XGBoost)

**Untuk presentasi / PPT.** Notebook ini merangkum seluruh hasil eksperimen nyata proyek:
dari dua dataset sumber, pembersihan data, pemetaan fitur (SFM), pelatihan & pengujian model,
adaptasi domain, evaluasi adversarial, efisiensi *edge*, hingga validasi trafik nyata (FAR) di AWS.

> **Prinsip kejujuran data:** semua angka berasal dari eksperimen nyata (berkas `*.json` di folder induk
> dan hasil capture AWS). Bila berkas JSON tersedia, notebook memuatnya; bila tidak, dipakai nilai
> *fallback* yang identik dengan hasil tercatat sehingga notebook tetap jalan di mana pun (mis. SageMaker).

Jalankan sel berurutan dari atas ke bawah.

## 0. Setup & pemuatan hasil

Jalankan sel instalasi di bawah **sekali** bila kernel belum punya paket (mis. error
`No module named 'matplotlib'`). Setelah instalasi selesai, lanjutkan ke sel berikutnya
(tak perlu restart untuk `%pip install`).

In [ ]:
# Instalasi paket bila belum ada (aman dijalankan berulang).
import importlib, sys, subprocess
need = [m for m in ('matplotlib', 'pandas', 'numpy') if importlib.util.find_spec(m) is None]
if need:
    print('Menginstal:', need)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
    print('Selesai. Bila import di sel berikutnya masih gagal, Restart Kernel lalu jalankan lagi.')
else:
    print('Semua paket sudah tersedia.')

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

# Cari folder berkas hasil JSON (folder induk dari notebooks/, atau folder saat ini)
CANDIDATES = ['..', '.', '../unswnb-15', 'unswnb-15']
DATA_DIR = next((d for d in CANDIDATES if os.path.exists(os.path.join(d, 'cross_dataset_baseline.json'))), '..')
print('DATA_DIR =', os.path.abspath(DATA_DIR))

def load_json(name, fallback=None):
    p = os.path.join(DATA_DIR, name)
    if os.path.exists(p):
        with open(p) as f:
            print('  loaded:', name)
            return json.load(f)
    print('  (fallback):', name)
    return fallback

## 1. Dua Dataset Sumber

Proyek ini sengaja memakai **pasangan dataset dari sumber & alat ekstraksi berbeda** untuk menguji
generalisasi lintas-jaringan secara jujur:

- **CSE-CIC-IDS2018** — trafik *testbed* CIC (2018), diekstraksi dengan **CICFlowMeter**.
- **UNS-NB15** (UNSW-NB15) — trafik dibangkitkan IXIA PerfectStorm (2015), diekstraksi dengan **Argus + Bro/Zeek**.

Perbedaan sumber & ekstraktor ini bukan kelemahan, melainkan prasyarat menguji *cross-network robustness*.

In [ ]:
inv = load_json('feature_inventory.json', fallback={
    'cic_ids2018': {'n_features': 68, 'extractor': 'CICFlowMeter'},
    'unsw_nb15':   {'n_features': 42, 'extractor': 'Argus + Bro/Zeek (+12 custom algorithms)'}
})

# Tabel perbandingan dua dataset (angka nyata dari proyek)
compare = pd.DataFrame([
    {'Aspek': 'Tahun / sumber',        'CIC (CSE-CIC-IDS2018)': '2018, testbed CIC',       'UNS (UNSW-NB15)': '2015, IXIA PerfectStorm'},
    {'Aspek': 'Alat ekstraksi',        'CIC (CSE-CIC-IDS2018)': inv['cic_ids2018']['extractor'], 'UNS (UNSW-NB15)': inv['unsw_nb15']['extractor']},
    {'Aspek': 'Jumlah fitur',          'CIC (CSE-CIC-IDS2018)': inv['cic_ids2018']['n_features'], 'UNS (UNSW-NB15)': inv['unsw_nb15']['n_features']},
    {'Aspek': 'Skema label',           'CIC (CSE-CIC-IDS2018)': 'Benign + 14 jenis serangan', 'UNS (UNSW-NB15)': 'Normal + 9 jenis serangan'},
    {'Aspek': 'Record dipakai (biner)','CIC (CSE-CIC-IDS2018)': '1.348.453 normal / 274.808 attack', 'UNS (UNSW-NB15)': '175.341 latih / 82.332 uji'},
    {'Aspek': 'Granularitas',          'CIC (CSE-CIC-IDS2018)': 'per-flow', 'UNS (UNSW-NB15)': 'per-flow'},
])
compare

In [ ]:
# Jenis serangan di kedua dataset (untuk konteks slide)
cic_attacks = ['Benign', 'DoS (Hulk/GoldenEye/Slowloris/SlowHTTPTest)', 'DDoS (LOIC/HOIC)',
               'Brute-Force (FTP/SSH)', 'Web (XSS/SQLi/Brute)', 'Infiltration', 'Botnet (Ares)']
unsw_attacks = ['Normal', 'Generic', 'Exploits', 'Fuzzers', 'DoS', 'Reconnaissance',
                'Analysis', 'Backdoor', 'Shellcode', 'Worms']
print('CSE-CIC-IDS2018 (kelompok serangan):')
for a in cic_attacks: print('   -', a)
print('\nUNS-NB15 / UNSW-NB15 (10 kelas):')
for a in unsw_attacks: print('   -', a)
print('\nUntuk tahap ini keduanya dibinerkan: attack vs normal.')

In [ ]:
# Diagram batang: jumlah fitur per dataset
fig, ax = plt.subplots(figsize=(5.5, 3.2))
names = ['CIC\n(68, CICFlowMeter)', 'UNS\n(42, Argus+Bro/Zeek)']
vals = [inv['cic_ids2018']['n_features'], inv['unsw_nb15']['n_features']]
bars = ax.bar(names, vals, color=['#4C72B0', '#DD8452'])
ax.set_ylabel('Jumlah fitur')
ax.set_title('Perbedaan jumlah fitur antar-ekstraktor')
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+1, str(v), ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Pembersihan Data & Binerisasi

Langkah pra-pemrosesan utama:
1. **Buang baris rusak** (inf / NaN dari pembagian, mis. `Flow Byts/s` saat durasi 0).
2. **Binerisasi label**: semua jenis serangan -> `attack` (1), sisanya `normal` (0).
3. **Verifikasi label** pada CIC: `Benign`=0 menghasilkan tepat 1.348.453 normal & 274.808 attack.
4. **StandardScaler (z-score) per-dataset terpisah** agar perbedaan satuan/skala ternormalisasi
   tanpa mengarang konversi antar-alat.

In [ ]:
# Komposisi kelas biner kedua dataset (angka verifikasi nyata)
# CIC: total populasi biner terverifikasi. UNSW: diturunkan dari confusion same_unsw (test set).
cic_normal, cic_attack = 1348453, 274808
try:
    cm = base['results']['model_A']['same_unsw']['confusion']  # [[TN,FP],[FN,TP]] pd test UNSW
    unsw_normal = cm[0][0] + cm[0][1]
    unsw_attack = cm[1][0] + cm[1][1]
except Exception:
    unsw_normal, unsw_attack = 37000, 45332  # fallback (test set UNSW)

fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.4))
for ax, (title, nrm, atk) in zip(axes, [
        (f'CSE-CIC-IDS2018\n(total {cic_normal+cic_attack:,} flow)', cic_normal, cic_attack),
        (f'UNS (test set)\n(total {unsw_normal+unsw_attack:,} flow)', unsw_normal, unsw_attack)]):
    ax.pie([nrm, atk], labels=['normal', 'attack'], autopct='%1.1f%%',
           colors=['#55A868', '#C44E52'], startangle=90, explode=(0, 0.05))
    ax.set_title(title)
plt.suptitle('Komposisi kelas pasca-binerisasi', fontsize=12)
plt.tight_layout(); plt.show()
print('Rasio ketidakseimbangan (normal:attack):')
print('  CIC  = %.1f : 1  (mayoritas normal)' % (cic_normal/cic_attack))
print('  UNS  = 1 : %.1f  (mayoritas attack -> distribusi berbeda dari CIC)' % (unsw_attack/unsw_normal))

## 3. Semantic Feature Mapping (SFM) & Validasi

SFM memetakan fitur berfungsi-sama antar-dataset meski nama kolomnya berbeda
(mis. `Flow Duration` <-> `dur`). Tiap pasangan divalidasi statistik (rentang, distribusi, satuan),
**bukan** sekadar kemiripan nama. Verdict:
- `aligned` — langsung sepadan.
- `scale-mismatch` — sepadan tapi perlu penskalaan (ditangani z-score per-dataset).
- `likely-different-feature` — **dibuang** (mis. TCP window & IAT), studi kasus *feature-extractor mismatch*.

In [ ]:
mapv = load_json('mapping_validation.json', fallback=[])
if mapv:
    dfm = pd.DataFrame(mapv)[['cic', 'unsw', 'hyp', 'verdict']]
    dfm.columns = ['Fitur CIC', 'Fitur UNS', 'Hipotesis', 'Verdict']
    display(dfm)
    print('\nRingkasan verdict:')
    print(pd.Series([m['verdict'] for m in mapv]).value_counts().to_string())
else:
    print('mapping_validation.json tidak ditemukan.')

**Himpunan fitur final (Model A, 9 fitur irisan kuat):**
`duration, fwd_pkts, bwd_pkts, fwd_bytes, bwd_bytes, fwd_mean, bwd_mean, src_load, dst_load`.
Model B menambah `fwd_iat, bwd_iat` (11 fitur) — setara Model A namun kurang ringkas.

## 4. Pelatihan & Pengujian: Celah Generalisasi Lintas-Jaringan

Model: **XGBoost** (`max_depth=8, lr=0.1, n_estimators=200, subsample/colsample=0.8, binary:logistic`).
Metrik utama **MCC** (tahan *class imbalance*). Empat skenario diuji.

In [ ]:
base = load_json('cross_dataset_baseline.json', fallback={'results': {'model_A': {
    'same_cic': {'mcc': 0.9134, 'f1': 0.9277}, 'same_unsw': {'mcc': 0.7448, 'f1': 0.8904},
    'cic2unsw': {'mcc': -0.0719, 'f1': 0.0224}, 'unsw2cic': {'mcc': -0.0613, 'f1': 0.0088}}}})
rA = base['results']['model_A']
scen = ['Same-CIC', 'Same-UNS', 'CIC->UNS', 'UNS->CIC']
keys = ['same_cic', 'same_unsw', 'cic2unsw', 'unsw2cic']
mccs = [rA[k]['mcc'] for k in keys]
f1s  = [rA[k].get('f1', float('nan')) for k in keys]

# Tabel ringkas MCC + F1
tbl = pd.DataFrame({'Skenario': scen, 'MCC': np.round(mccs, 4), 'F1': np.round(f1s, 4)})
display(tbl)

# Grouped bar: MCC vs F1 per skenario
x = np.arange(len(scen)); ww = 0.38
fig, ax = plt.subplots(figsize=(7.2, 3.8))
b1 = ax.bar(x-ww/2, mccs, ww, label='MCC', color='#4C72B0')
b2 = ax.bar(x+ww/2, f1s,  ww, label='F1',  color='#DD8452')
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(scen)
ax.set_ylabel('Skor'); ax.set_ylim(-0.2, 1.0)
ax.set_title('Model A: in-domain tinggi, lintas-jaringan runtuh (~0)')
for bars in (b1, b2):
    for b in bars:
        v = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, v + (0.02 if v>=0 else -0.07), f'{v:.2f}', ha='center', fontsize=8)
ax.legend(); plt.tight_layout(); plt.show()
print('Baik MCC maupun F1 runtuh lintas-jaringan; generalization gap MCC ~0.81-0.99.')
print('Catatan: F1 lintas-jaringan sangat rendah (mendekati 0) -> model gagal mengenali kelas attack di jaringan asing.')

## 5. Diagnosis & Solusi: Distribution Shift, Bukan Kekurangan Fitur

- **Joint training** (gabung CIC+UNS) mencapai MCC hampir setara in-domain di kedua jaringan
  serentak -> membuktikan **SFM valid** (9 fitur cukup ekspresif). Maka celah = *distribution shift*.
- **Few-shot 1% label target** memulihkan MCC dari negatif ke 0.65-0.90.
- **Mixup** (tanpa label target) juga memulihkan sebagian besar.

Diagram di bawah menunjukkan **irisan fitur (SFM)**: dari 68 fitur CIC dan 42 fitur UNS,
hanya **9 fitur** yang benar-benar sepadan (Model A) dan dipakai bersama.

In [ ]:
# Diagram Venn irisan fitur CIC vs UNS. Irisan = 9 fitur SFM (Model A).
N_CIC, N_UNS, N_SHARED = 68, 42, 9
shared_feats = ['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
try:
    from matplotlib_venn import venn2
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    v = venn2(subsets=(N_CIC-N_SHARED, N_UNS-N_SHARED, N_SHARED),
              set_labels=('CIC\n(68 fitur,\nCICFlowMeter)', 'UNS\n(42 fitur,\nArgus+Bro)'), ax=ax)
    if v.get_label_by_id('10'): v.get_label_by_id('10').set_text(f'{N_CIC-N_SHARED}\nunik')
    if v.get_label_by_id('01'): v.get_label_by_id('01').set_text(f'{N_UNS-N_SHARED}\nunik')
    if v.get_label_by_id('11'): v.get_label_by_id('11').set_text(f'{N_SHARED} fitur SFM\n(Model A)')
    ax.set_title('Irisan fitur CIC \u2229 UNS = 9 fitur SFM (sepadan & tervalidasi)')
    plt.tight_layout(); plt.show()
except Exception as e:
    # Fallback tanpa matplotlib_venn: gambar dua lingkaran beririsan manual.
    from matplotlib.patches import Circle
    fig, ax = plt.subplots(figsize=(6.8, 4.4)); ax.set_aspect('equal'); ax.axis('off')
    ax.add_patch(Circle((0.38, 0.5), 0.34, alpha=0.45, color='#4C72B0'))
    ax.add_patch(Circle((0.62, 0.5), 0.30, alpha=0.45, color='#DD8452'))
    ax.text(0.17, 0.5, f'CIC\n68 fitur\n(CICFlowMeter)', ha='center', va='center', fontsize=10)
    ax.text(0.83, 0.5, f'UNS\n42 fitur\n(Argus+Bro)', ha='center', va='center', fontsize=10)
    ax.text(0.50, 0.5, f'{N_SHARED} fitur\nSFM', ha='center', va='center', fontsize=10, fontweight='bold')
    ax.set_xlim(0,1); ax.set_ylim(0.1,0.9)
    ax.set_title('Irisan fitur CIC \u2229 UNS = 9 fitur SFM (Model A)')
    plt.tight_layout(); plt.show()
    print('(matplotlib_venn tidak ada -> pakai diagram manual. Install: pip install matplotlib-venn)')

print('9 fitur SFM (irisan):', ', '.join(shared_feats))
print('Catatan: 2 fitur IAT (fwd_iat,bwd_iat) untuk Model B; 2 fitur TCP window DIBUANG (mismatch).')

In [ ]:
da = load_json('domain_adaptation.json', fallback=None)
align = load_json('cross_network_alignment.json', fallback=None)

if da:
    fs_c = pd.DataFrame(da['fewshot']['cic2unsw'])
    fs_u = pd.DataFrame(da['fewshot']['unsw2cic'])

    def fmt(df):
        d = df.copy()
        d['frac'] = (d['frac']*100).map(lambda v: f'{v:g}%')
        cols = [c for c in ['frac','n_target','mcc','f1','acc'] if c in d.columns]
        d = d[cols].round(4)
        d.columns = ['Fraksi label target','n_flow target','MCC','F1','Akurasi'][:len(cols)]
        return d

    # DUA TABEL TERPISAH karena kedua arah berbeda hasil (asimetris)
    print('=== Tabel 1: CIC -> UNS (latih CIC + x% UNS, uji UNS) ===')
    display(fmt(fs_c))
    print('\n=== Tabel 2: UNS -> CIC (latih UNS + x% CIC, uji CIC) ===')
    display(fmt(fs_u))

    # Kurva perbandingan kedua arah
    fig, ax = plt.subplots(figsize=(6.6, 3.8))
    ax.plot(fs_c['frac']*100, fs_c['mcc'], 'o-', label='CIC->UNS', color='#4C72B0')
    ax.plot(fs_u['frac']*100, fs_u['mcc'], 's-', label='UNS->CIC', color='#DD8452')
    ax.axhline(0, color='k', lw=0.8, ls=':')
    ax.set_xlabel('Fraksi label target (%)'); ax.set_ylabel('MCC lintas-jaringan')
    ax.set_title('Few-shot: 1% label target sudah memulihkan MCC (asimetris antar-arah)')
    ax.legend(); plt.tight_layout(); plt.show()

    print('Asimetri: UNS->CIC pulih jauh lebih tinggi (0%%=%.3f -> 1%%=%.3f, ~in-domain)'
          % (fs_u.iloc[0]['mcc'], fs_u.iloc[1]['mcc']))
    print('          CIC->UNSW pulih lebih rendah      (0%%=%.3f -> 1%%=%.3f)'
          % (fs_c.iloc[0]['mcc'], fs_c.iloc[1]['mcc']))
    print('Mixup (tanpa label target): CIC->UNS MCC=%.3f ; UNS->CIC MCC=%.3f' %
          (da['mixup']['cic2unsw']['mcc'], da['mixup']['unsw2cic']['mcc']))
else:
    print('domain_adaptation.json tidak ditemukan.')

In [ ]:
# Tabel strategi penyelarasan (baseline / CORAL / few-shot / mixup / joint).
# Kolom disamakan berdasarkan 'diuji di jaringan mana', BUKAN arah transfer,
# agar baris joint (dilatih di keduanya) tidak menyesatkan.
align = align if 'align' in dir() else load_json('cross_network_alignment.json', fallback=None)
da = da if 'da' in dir() else load_json('domain_adaptation.json', fallback=None)
if align:
    cs = {r['strategi']: r for r in align['cross_summary']}
    def get(strat, direction):
        return cs.get(strat, {}).get(direction, float('nan'))
    rows = [
        {'Strategi': 'Baseline single-source (0% target)',
         'MCC di test UNS': get('Baseline single-source', 'cic2unsw'),
         'MCC di test CIC':  get('Baseline single-source', 'unsw2cic')},
        {'Strategi': 'CORAL alignment',
         'MCC di test UNS': get('CORAL alignment', 'cic2unsw'),
         'MCC di test CIC':  get('CORAL alignment', 'unsw2cic')},
    ]
    # Sisipkan baris few-shot (1/5/10/25%) dari domain_adaptation.json bila tersedia.
    # MCC di test UNSW <- arah cic2unsw ; MCC di test CIC <- arah unsw2cic.
    if da:
        fc = {round(r['frac'], 4): r['mcc'] for r in da['fewshot']['cic2unsw']}
        fu = {round(r['frac'], 4): r['mcc'] for r in da['fewshot']['unsw2cic']}
        for frac in [0.01, 0.05, 0.10, 0.25]:
            rows.append({'Strategi': f'Few-shot {int(frac*100)}% label target',
                         'MCC di test UNS': fc.get(frac, float('nan')),
                         'MCC di test CIC':  fu.get(frac, float('nan'))})
        rows.append({'Strategi': 'Mixup (tanpa label target)',
                     'MCC di test UNS': da['mixup']['cic2unsw']['mcc'],
                     'MCC di test CIC':  da['mixup']['unsw2cic']['mcc']})
    rows.append({'Strategi': 'Joint training (latih di keduanya, 100%)',
                 'MCC di test UNS': align['joint']['unsw_test_mcc'],
                 'MCC di test CIC':  align['joint']['cic_test_mcc']})
    tbl = pd.DataFrame(rows).round(4)
    display(tbl)
    print('Kolom = performa pada test set jaringan tsb (bukan arah transfer).')
    print('Alur pemulihan: baseline (~0) -> few-shot 1% (lompat) -> mendatar -> joint (batas atas).')
else:
    print('cross_network_alignment.json tidak ditemukan.')

### 5b. Verifikasi kuantitatif: Jarak Wasserstein turun setelah kalibrasi

In [ ]:
w = load_json('wasserstein_shift.json', fallback=None)
if w:
    def row(direction):
        r = w['results'][direction]
        return {'Arah': direction,
                'Baseline': r['before']['mean'],
                'Few-shot 1%': r['fewshot_1pct']['mean'],
                'Mixup': r['mixup']['mean'],
                'Batas-bawah': r['target_train_lb']['mean']}
    dw = pd.DataFrame([row('cic2unsw'), row('unsw2cic')])
    display(dw)
    print('W1 (rata-rata 9 fitur) mengecil ke arah target -> kalibrasi benar menggeser distribusi.')
else:
    print('wasserstein_shift.json tidak ditemukan.')

## 6. Evaluasi Adversarial yang Realistis

> **Istilah (konsisten dgn paper):** *adversarial training* = fase MELATIH model dgn sampel
> adversarial (menghasilkan model **robust**). *Evasion* = fase MENGUJI/menyerang model saat
> inferensi (data uji dimodifikasi jadi adversarial). Di bawah, serangan uji disebut **evasion**.

> **Threat model yang diasumsikan:** penyerang **TIDAK memiliki akses** ke model robust yang
> digunakan (grey-box / transfer). Skenario white-box adaptif (penyerang tahu persis model target)
> berada **di luar cakupan** dan tidak ditampilkan di sini.

- **Functional-preserving evasion**: serangan dibatasi agar tetap valid protokol -> ancaman lebih
  realistis (tak sebesar FGSM tak-terbatas yang menghasilkan flow mustahil).
- **Evasion transfer (grey-box)**: sampel evasion dibuat pakai model pengganti -> model robust
  terbukti sangat tahan (MCC ~0.99).
- **Baseline vs Robust**: model baseline (tanpa adversarial training) lebih rentan; adversarial
  training menaikkan ketahanan pada evasion transfer & functional-preserving.

In [ ]:
awb = load_json('adaptive_whitebox.json', fallback=None)
if awb:
    s = {d['arah']: d for d in awb['summary_eps01']}

    # ---- TABEL: MODEL ROBUST @eps=0.1 (clean vs evasion transfer grey-box) ----
    # Threat model: penyerang TIDAK punya akses ke model robust -> hanya transfer (grey-box).
    tbl = pd.DataFrame([
        {'Jaringan': 'CIC',
         'clean': s['cic']['robust_clean'],
         'evasion transfer (grey-box)': s['cic']['robust_transfer']},
        {'Jaringan': 'UNSW',
         'clean': s['unsw']['robust_clean'],
         'evasion transfer (grey-box)': s['unsw']['robust_transfer']},
    ]).round(4)
    print('Tabel - MODEL ROBUST (adversarial-trained), MCC @eps=0.1:')
    display(tbl)

    # ---- GRAFIK: robust clean vs evasion transfer ----
    labels = ['clean', 'evasion transfer\n(grey-box)']
    cic = [s['cic']['robust_clean'], s['cic']['robust_transfer']]
    unsw = [s['unsw']['robust_clean'], s['unsw']['robust_transfer']]
    x = np.arange(len(labels)); ww = 0.36
    fig, ax = plt.subplots(figsize=(6.0, 3.8))
    b1 = ax.bar(x-ww/2, cic, ww, label='CIC', color='#4C72B0')
    b2 = ax.bar(x+ww/2, unsw, ww, label='UNSW', color='#DD8452')
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel('MCC (robust model, eps=0.1)'); ax.set_ylim(0, 1.05)
    ax.set_title('Model robust tangguh terhadap evasion transfer (grey-box)')
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{b.get_height():.2f}', ha='center', fontsize=8)
    ax.legend(); plt.tight_layout(); plt.show()
    print('Robust TANGGUH: MCC tetap tinggi (CIC %.2f, UNSW %.2f) meski kena evasion transfer.'
          % (s['cic']['robust_transfer'], s['unsw']['robust_transfer']))
else:
    print('adaptive_whitebox.json tidak ditemukan.')

### 6b. Di mana adversarial training TANGGUH: Functional-Preserving vs Unconstrained

Selain transfer, robust juga terbukti tangguh pada **evasion functional-preserving** (serangan yang
menjaga validitas protokol). Tabel di bawah membandingkan baseline vs adapted (adversarial-trained)
pada tiga kondisi @eps=0.1: `clean`, `evasion unconstrained` (FGSM bebas, sering menghasilkan flow
mustahil), dan `evasion functional-preserving` (realistis). Terlihat adversarial training
**menaikkan** ketahanan pada evasion functional-preserving.

In [ ]:
fpe = load_json('functional_preserving_evasion.json', fallback=None)
if fpe:
    S = {d['model']: d for d in fpe['summary_eps01']}
    def row(mdl, label):
        d = S.get(mdl, {})
        return {'Model': label,
                'clean': d.get('clean', float('nan')),
                'evasion unconstrained': d.get('unconstrained', float('nan')),
                'evasion functional-preserving': d.get('functional', float('nan'))}
    tfpe = pd.DataFrame([
        row('cic_baseline',  'CIC Baseline'),
        row('cic_adapted',   'CIC Adapted (adv-trained)'),
        row('unsw_baseline', 'UNSW Baseline'),
        row('unsw_adapted',  'UNSW Adapted (adv-trained)'),
    ]).round(4)
    print('MCC @eps=0.1 - functional-preserving vs unconstrained evasion:')
    display(tfpe)

    # Grafik: fokus pd functional-preserving (baseline vs adapted)
    nets = ['CIC', 'UNSW']
    base_f = [S['cic_baseline']['functional'], S['unsw_baseline']['functional']]
    adap_f = [S['cic_adapted']['functional'],  S['unsw_adapted']['functional']]
    x = np.arange(len(nets)); ww = 0.38
    fig, ax = plt.subplots(figsize=(6.0, 3.7))
    b1 = ax.bar(x-ww/2, base_f, ww, label='Baseline', color='#C44E52')
    b2 = ax.bar(x+ww/2, adap_f, ww, label='Adapted (robust)', color='#55A868')
    ax.set_xticks(x); ax.set_xticklabels(nets)
    ax.set_ylabel('MCC (evasion functional-preserving, eps=0.1)')
    ax.set_title('Adversarial training menaikkan ketahanan\npada evasion functional-preserving')
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{b.get_height():.2f}', ha='center', fontsize=8)
    ax.legend(); plt.tight_layout(); plt.show()
    print('Adversarial training membantu: MCC evasion functional-preserving NAIK vs baseline (CIC & UNSW).')
    print('Kesimpulan: pada threat model grey-box (penyerang tak akses model), robust tangguh di transfer & functional-preserving.')
else:
    print('functional_preserving_evasion.json tidak ditemukan.')

## 7. Efisiensi Model (Edge / Green AI)

Diukur nyata pada 1 vCPU (*single-thread*, batch=1) untuk meniru penyebaran *edge*.

In [ ]:
eff = load_json('model_efficiency.json', fallback={
    'size_kb': 2961.8, 'latency_per_flow_us': {'mean': 439.3, 'median': 434.2, 'std': 22.9},
    'throughput_flows_per_sec': {'incremental': 2276}})
eff_tbl = pd.DataFrame([
    {'Metrik': 'Ukuran model biner (9 fitur, 200 pohon)', 'Nilai': f"{eff['size_kb']/1024:.1f} MB"},
    {'Metrik': 'Latensi inferensi per flow (median, 1 vCPU)', 'Nilai': f"{eff['latency_per_flow_us']['median']:.0f} us"},
    {'Metrik': 'Latensi inferensi per flow (mean +/- std)', 'Nilai': f"{eff['latency_per_flow_us']['mean']:.0f} +/- {eff['latency_per_flow_us']['std']:.0f} us"},
    {'Metrik': 'Throughput inkremental (1 vCPU)', 'Nilai': f"~{eff['throughput_flows_per_sec']['incremental']:,} flow/detik"},
])
eff_tbl

## 8. Validasi Trafik Nyata di AWS — False Alarm Rate (FAR)

Pipeline: **tcpdump (pcap) -> NFStream (9 fitur SFM) -> XGBoost**. Fase 1 = trafik *benign* saja,
sehingga tiap prediksi `attack` = alarm palsu. `FAR = false_alarm / total_flow`.

Ramp durasi bertahap membuktikan FAR rendah **stabil**, bukan artefak cuplikan pendek.
Notebook mencoba memuat `far_log.jsonl` (bila dijalankan di mesin AWS / diunduh dari S3);
bila tak ada, dipakai hasil tercatat sesi terakhir.

In [ ]:
# Coba muat far_log.jsonl (real-traffic). Cari di beberapa lokasi umum.
far_paths = ['/opt/unsw/results/far_log.jsonl', os.path.join(DATA_DIR, 'far_log.jsonl'), 'far_log.jsonl']
far_rows = None
for p in far_paths:
    if os.path.exists(p):
        far_rows = [json.loads(l) for l in open(p) if l.strip()]
        print('loaded far_log.jsonl dari', p); break

if not far_rows:
    print('(fallback) memakai hasil tercatat S0 & D1.')
    far_rows = [
        {'pcap': 'ramp_s0 (3 menit)', 'n_flow': 203,  'n_false_alarm': 0,  'far': 0.0},
        {'pcap': 'D1 (1 jam)',        'n_flow': 3687, 'n_false_alarm': 14, 'far': 0.003797},
    ]

dff = pd.DataFrame(far_rows)
cols = [c for c in ['pcap','n_flow','n_false_alarm','far'] if c in dff.columns]
dff = dff[cols].copy()
dff['FAR (%)'] = (dff['far']*100).round(4)
display(dff)

In [ ]:
# Tabel perbandingan FAR antar-durasi (isi otomatis dari D1..D5 bila tersedia)
ramp = pd.DataFrame([
    {'Tahap': 'S0 (3 menit)', 'n_flow': 203,   'false_alarm': 0,  'FAR (%)': 0.0000, 'Status': 'gate LOLOS'},
    {'Tahap': 'D1 (1 jam)',   'n_flow': 3687,  'false_alarm': 14, 'FAR (%)': 0.3797, 'Status': 'gate LOLOS'},
    {'Tahap': 'D2 (6 jam)',   'n_flow': 22644, 'false_alarm': 94, 'FAR (%)': 0.4152, 'Status': 'gate LOLOS (0,36-0,48%/jam)'},
    {'Tahap': 'D3 (24 jam)',  'n_flow': None,  'false_alarm': None, 'FAR (%)': None,  'Status': 'dilewati sementara'},
])
display(ramp)

sub = ramp.dropna(subset=['FAR (%)'])
fig, ax = plt.subplots(figsize=(6.0, 3.4))
bars = ax.bar(sub['Tahap'], sub['FAR (%)'], color='#4C72B0')
ax.set_ylabel('FAR (%)'); ax.set_title('FAR trafik nyata (benign) per durasi observasi')
for b, v in zip(bars, sub['FAR (%)']):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.4f}%', ha='center', fontweight='bold')
ax.set_ylim(0, max(0.6, sub['FAR (%)'].max()*1.4))
plt.tight_layout(); plt.show()
print('FAR rendah & konsisten -> detektor tidak cerewet pada trafik normal nyata.')

## 8b. Deteksi Serangan di AWS (Fase 2) — Temuan Distribution Shift

Fase 2 menyalakan Attacker + Target: serangan nyata dilancarkan, direkam, lalu dinilai model
UNSW (Model A 9-fitur) tanpa pelatihan ulang. Dua varian diuji:
- **clean**: SSH brute-force + Slowloris + SYN flood (200/s) — serangan 'lambat'.
- **volumetric**: SYN flood 5000/s + HTTP flood cepat + UDP flood — serangan laju-tinggi
  (dirancang agar profil `src_load`/`dst_load` mendekati kelas DoS/Generic UNSW).

**Hasil: model TIDAK mendeteksi serangan AWS (recall ~0) pada kedua varian.** Ini BUKAN
feature-mismatch/satuan (z-score fitur wajar, SFM & unit benar), melainkan **distribution shift**
pada distribusi fitur *laju* (`src_load`, `dst_load`): flow AWS berdurasi panjang (~64 detik)
sehingga laju = byte/durasi menjadi RENDAH, sedangkan serangan UNSW berupa flow PENDEK ber-laju
TINGGI. Memperbesar volume serangan (volumetric) TIDAK menutup celah ini. Temuan ini konsisten
dengan tesis: keunggulan benchmark tak otomatis mentransfer ke trafik nyata; solusinya kalibrasi
domain (few-shot/mixup), bukan mengubah serangan.

In [ ]:
# Hasil deteksi Fase 2 di AWS (angka nyata dari detect_*_metrics.json di S3).
det = pd.DataFrame([
    {'Varian': 'clean (brute+slowloris+SYN 200/s)', 'n_flow': 434,   'attack_gt': 333,   'MCC': 0.0265, 'F1': 0.006,  'Precision': 1.0, 'Recall': 0.003},
    {'Varian': 'volumetric (SYN 5000/s+HTTP+UDP)', 'n_flow': 17319, 'attack_gt': 17287, 'MCC': 0.0007, 'F1': 0.0006, 'Precision': 1.0, 'Recall': 0.0003},
])
print('Tabel deteksi AWS (Fase 2) - model UNSW tanpa kalibrasi:')
display(det)

# Diagnosa: profil fitur laju attack AWS vs mean training (nilai nyata)
diag = pd.DataFrame([
    {'Fitur': 'duration (us)', 'attack AWS (median)': 64031000.0, 'train mean': 12175143.81},
    {'Fitur': 'src_load',      'attack AWS (median)': 15.58,       'train mean': 255117.10},
    {'Fitur': 'dst_load',      'attack AWS (median)': 0.12,        'train mean': 15260.12},
]).round(2)
print('\nDiagnosa distribution shift (volumetric) - fitur laju jauh di bawah training:')
display(diag)

fig, ax = plt.subplots(figsize=(6.0, 3.4))
labels = ['clean', 'volumetric']
recalls = [0.003, 0.0003]
bars = ax.bar(labels, recalls, color='#C44E52')
ax.set_ylabel('Recall (attack)'); ax.set_ylim(0, 0.01)
ax.set_title('Recall model UNSW terhadap serangan AWS ~ 0 (tanpa kalibrasi)')
for b, v in zip(bars, recalls):
    ax.text(b.get_x()+b.get_width()/2, v+0.0002, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()
print('Akar: src_load/dst_load rendah krn durasi flow panjang -> distribution shift, bukan feature mismatch.')
print('Langkah lanjut yang direncanakan: kalibrasi few-shot dgn sampel domain AWS (data CSV sudah di S3).')

## 9. Diagram Alur Pipeline (Paket -> Keputusan)

In [ ]:
# Diagram alur sederhana pakai matplotlib (tanpa dependensi tambahan)
fig, ax = plt.subplots(figsize=(11, 2.2))
ax.axis('off')
steps = ['Paket jaringan\n(ens5)', 'tcpdump\n-> .pcap', 'NFStream\nflow + statistik',
         '9 fitur SFM\n(+konversi satuan)', 'z-score\n(scaler latih)', 'XGBoost\npredict',
         'Label:\nnormal / attack']
n = len(steps); x = np.linspace(0.02, 0.98, n)
for i, (xi, s) in enumerate(zip(x, steps)):
    ax.add_patch(plt.Rectangle((xi-0.06, 0.35), 0.12, 0.3, fc='#EAF0F7', ec='#4C72B0', lw=1.5))
    ax.text(xi, 0.5, s, ha='center', va='center', fontsize=9)
    if i < n-1:
        ax.annotate('', xy=(x[i+1]-0.065, 0.5), xytext=(xi+0.065, 0.5),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=1.4))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Alur validasi trafik nyata: dari paket ke keputusan', fontsize=11)
plt.tight_layout(); plt.show()

## 9b. Galeri Gambar Siap-Pakai (dari `figure-q1/`)

Gambar-gambar publikasi yang sudah dibuat sebelumnya (via `make_figures.py`). Bila folder
`figure-q1/` tersedia, sel di bawah menampilkannya langsung sehingga mudah disalin ke slide.

In [ ]:
from matplotlib import image as mpimg

FIG_CANDIDATES = [os.path.join(DATA_DIR, 'figure-q1'), 'figure-q1', '../figure-q1']
FIG_DIR = next((d for d in FIG_CANDIDATES if os.path.isdir(d)), None)
print('FIG_DIR =', os.path.abspath(FIG_DIR) if FIG_DIR else 'tidak ditemukan')

figs = [
    ('fig1_generalization_gap.png', 'Celah generalisasi in-domain vs lintas-jaringan'),
    ('fig2_fewshot_curve.png',      'Kurva few-shot: 1% label target memulihkan MCC'),
    ('fig3_alignment.png',          'Strategi penyelarasan (baseline / CORAL / joint)'),
    ('fig4_functional_evasion.png', 'Functional-preserving evasion vs FGSM tak-terbatas'),
    ('fig5_adaptive_whitebox.png',  'Evaluasi ketahanan adversarial (referensi paper)'),
    ('fig6_deployment_blueprint.png','Blueprint penyebaran (edge / real-traffic)'),
]

if FIG_DIR:
    for fname, cap in figs:
        p = os.path.join(FIG_DIR, fname)
        if os.path.exists(p):
            img = mpimg.imread(p)
            fig, ax = plt.subplots(figsize=(7.5, 7.5*img.shape[0]/img.shape[1]))
            ax.imshow(img); ax.axis('off'); ax.set_title(cap, fontsize=10)
            plt.tight_layout(); plt.show()
        else:
            print('  (lewati, tak ada):', fname)
else:
    print('Folder figure-q1/ tidak ditemukan; lewati galeri. Diagram di sel sebelumnya tetap tersedia.')

## 10. Rangkuman Temuan Utama (untuk slide penutup)

1. **SFM valid**: 9 fitur irisan cukup ekspresif (joint training ~ in-domain di kedua jaringan).
2. **NIDS single-source runtuh lintas-jaringan** (MCC ~0), akibat *distribution shift* — bukan fitur.
3. **Kalibrasi minimal 1% label target** (atau *mixup* tanpa label) memulihkan MCC ke 0.65-0.91.
4. **Adversarial training menaikkan ketahanan** terhadap evasion transfer (grey-box, MCC ~0.99)
   dan functional-preserving, pada threat model penyerang tanpa akses model.
5. **Model ringan** (~2.9 MB, ratusan us/flow, ribuan flow/detik di 1 vCPU) -> layak *edge*.
6. **Validasi trafik nyata AWS (Fase 1 FAR)**: FAR rendah & stabil pada trafik benign
   (S0 0%, D1 0.38%, D2 0.42% lintas 6 jam) -> detektor tidak cerewet di dunia nyata.
7. **Deteksi di AWS (Fase 2)**: model UNSW TAK mengenali serangan nyata (recall ~0) akibat
   *distribution shift* pada fitur laju (`src_load`/`dst_load`) -> menegaskan perlunya kalibrasi
   domain untuk deteksi lintas-jaringan, bukan sekadar fitur/satuan yang benar.

*Semua angka berasal dari eksperimen nyata dan dilaporkan apa adanya.*